# Day 2-03：ImageNet学習済みモデルを騙す

高解像度画像に対して、同じI-FGSMを適用します。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. 学習済みモデル

`timm`からImageNetの1000クラスを分類するモデルを読み込みます。リサイズと切り抜きはモデルの設定に合わせます。正規化はモデル内部で行い、攻撃対象の画素値を0〜1に保ちます。

In [ ]:
from jkj1a.classification import load_imagenet_classifier
import torch
import timm
from PIL import Image
from pathlib import Path
from urllib.request import urlretrieve

model_name = "resnet18.a1_in1k"
model, transform, labels = load_imagenet_classifier(model_name)
device = next(model.parameters()).device

## 2. 画像と予測

初回はサンプル画像をダウンロードします。自分の画像を使う場合は`image_path`を変更してください。元画像と、リサイズ・切り抜き後の画像を比較します。

In [ ]:
from jkj1a.classification import show_predictions

image_path = Path("data/day2/dog.jpg")
if not image_path.exists():
    image_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg", image_path)

# 元画像を確認する
image = Image.open(image_path).convert("RGB")

display(image)

# モデルに合わせて前処理し、バッチの軸を追加する
x = transform(image).unsqueeze(0).to(device)

show_predictions(model, x, labels, n=1)

with torch.no_grad():
    original_label = model(x).argmax(1)

print("変更前:", original_label.item(), labels[original_label.item()])

## 3. 予測クラスを変える

この例では変更前の予測クラスを基準に攻撃します。元画像の正解ラベルは与えていないので、測っているのは「予測が変わったか」です。変更前に正解していたことを確認して初めて、誤分類を誘発したと判断できます。

In [ ]:
from jkj1a.adversarial import ifgsm
from jkj1a.adversarial import show_attack

x_adv = ifgsm(model, x, original_label, eps=8/255, alpha=2/255, n_steps=10)

show_attack(model, x, x_adv, labels)

with torch.no_grad():
    attacked_label = model(x_adv).argmax(1)

print("予測が変わった:", (attacked_label != original_label).item())

## 4. 入力と設定を保存する

PNG保存では量子化が入ります。ここでは再現用に攻撃前後のテンソルをそのまま保存します。

In [ ]:
output = Path("results/day2-imagenet")
output.mkdir(parents=True, exist_ok=True)

# 画像テンソルと攻撃条件を保存する
torch.save({"model_name": model_name, "x": x.cpu(), "x_adv": x_adv.cpu(),
            "eps": 8/255, "alpha": 2/255, "n_steps": 10,
            "original_label": original_label.cpu(),
            "torch_version": torch.__version__, "timm_version": timm.__version__}, output / "attack.pt")

## 演習

1. 自分で選んだ画像3枚以上で実行し、変更前後の予測と画素の最大変化を記録してください。
2. 1枚を選び、見た目を確認しながら予測が変わる`eps`を探してください。
3. 別の目標クラスを指定してtargeted attackを行ってください。変更前にそのクラスでないことも確認してください。

In [ ]:
# ここにコードを書く